# 02 Econometric Models (Workbook-Driven)

This notebook estimates the workbook-driven specifications (M1--M7b) using the refreshed
processed panel produced by `01_data_processing.ipynb`.

Broad money is measured as broad money growth (annual %) (`broad_money_growth_pct`).


## Setup


In [40]:
from pathlib import Path
import sys
import importlib

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import MODEL_SELECTION_WORKBOOK, OUTPUTS_DIR, PROCESSED_PANEL_FILE, ensure_output_dirs
import src.model_contract as model_contract
import src.estimate_and_export as estimate_and_export

importlib.reload(model_contract)
importlib.reload(estimate_and_export)

build_workbook_model_catalog = model_contract.build_workbook_model_catalog
run_full_estimation_and_export = estimate_and_export.run_full_estimation_and_export

ensure_output_dirs()
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Load Processed Panel


In [41]:
panel = pd.read_csv(PROCESSED_PANEL_FILE)
panel.shape, int(panel['year'].min()), int(panel['year'].max())


((264, 18), 2000, 2023)

In [42]:
required = [
    'country', 'year', 'fdi_pct_gdp', 'broad_money_growth_pct',
    'inflation_gdp_deflator_pct', 'trade_pct_gdp', 'ln_gdppc', 'xr_dep_pct',
]
missing = [c for c in required if c not in panel.columns]
if missing:
    raise ValueError(
        f"Processed panel is missing required columns: {missing}. "
        "Re-run 01_data_processing.ipynb from the top to rebuild clean_panel.csv."
    )


## Build Workbook Catalog


In [43]:
catalog, workbook_tables = build_workbook_model_catalog(
    MODEL_SELECTION_WORKBOOK,
    panel_columns=panel.columns.tolist(),
)
catalog[['model_id','workbook_model','status','missing_reason','base_regressors','mapped_regressors']]


,model_id,workbook_model,status,missing_reason,base_regressors,mapped_regressors
0,M1_baseline_liquidity,M1 - Baseline liquidity,estimated,,"broad_money_growth_pct, inflation_gdp_deflator...","broad_money_growth_pct, inflation_gdp_deflator..."
1,M2_main_monetary_policy,M2 - Main monetary policy,estimated,,"broad_money_growth_pct, deposit_interest_rate_...","broad_money_growth_pct, deposit_interest_rate_..."
2,M3_lagged_main_model,M3 - Lagged main model,estimated,,"broad_money_growth_pct, deposit_interest_rate_...","broad_money_growth_pct_lag1, deposit_interest_..."
3,M4_real_interest_robustness,M4 - Real interest robustness,estimated,,"broad_money_growth_pct, real_interest_rate_pct...","broad_money_growth_pct, real_interest_rate_pct..."
4,M5_lending_rate_robustness,M5 - Lending rate robustness,estimated,,"broad_money_growth_pct, lending_interest_rate_...","broad_money_growth_pct, lending_interest_rate_..."
5,M6_tourism_robustnessa_from_M4_real_interest_r...,M6 - Tourism robustness,estimated,,"broad_money_growth_pct, real_interest_rate_pct...","broad_money_growth_pct, real_interest_rate_pct..."
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,M6 - Tourism robustness,estimated,,"broad_money_growth_pct, deposit_interest_rate_...","broad_money_growth_pct, deposit_interest_rate_..."
7,M7_human_capital_robustnessa_from_M4_real_inte...,M7 - Human capital robustness,estimated,,"broad_money_growth_pct, real_interest_rate_pct...","broad_money_growth_pct, real_interest_rate_pct..."
8,M7_human_capital_robustnessb_from_M2_main_mone...,M7 - Human capital robustness,estimated,,"broad_money_growth_pct, deposit_interest_rate_...","broad_money_growth_pct, deposit_interest_rate_..."


## Estimate And Export


In [44]:
estimated_models = catalog[catalog['status'].eq('estimated')].copy()
run_full_estimation_and_export(panel, estimated_models, MODEL_SELECTION_WORKBOOK, dependent='fdi_pct_gdp')


--- Starting Full Econometric Estimation & Export Pipeline ---
Estimating native specifications...


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.w

Running model sample loss audits...
Running common sample estimations...


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.w

Running model panel balance summaries...
Writing main output CSV files...
Computing descriptive stats and correlations...
Building diagnostic tables and regression detail files for individual models...
Building main regression table...
Running stepwise broad money sign decomposition...
Running trade collinearity robustness drops...
Writing bulk workbook outputs/model_outputs.xlsx...
Writing bulk workbook outputs/results_tables.xlsx...
--- Econometric Estimation & Export Pipeline Completed Successfully ---


## Quick Output Check


In [45]:
for name in [
    'model_coefficients.csv',
    'model_fit_stats.csv',
    'regression_table_main.csv',
    'correlation_matrix_main_variables.csv',
]:
    path = OUTPUTS_DIR / name
    print(name, 'exists' if path.exists() else 'MISSING')


model_coefficients.csv exists
model_fit_stats.csv exists
regression_table_main.csv exists
correlation_matrix_main_variables.csv exists
